# OpenClaw Lab 1: Overall Harness

Before running the lab, let's first say what OpenClaw is.

OpenClaw is a platform for running personal AI agents. In a full setup, an OpenClaw agent can connect to chat apps, local devices, tools, automations, and multiple workspaces. It can act as a personal assistant, a service-side agent, a channel bot, or a local agent runtime that you control.

For now, just think of **OpenClaw** as a powerful workbench for running AI agents. OpenClaw can do much more than this lab will cover, but we will begin with the part that matters most for understanding an agent turn: connecting a language model to files, command-line tools, session history, and memory notes.

At first, this may feel like a black box. You give the agent a task, and it produces an answer. This lab opens that box step by step.

In this lab, you will play the role of a TA preparing course lab materials for an AI Agent and LLM Serving course. The lab workspace contains a course brief, FAQ, rubric, release checklist, memory note, and one sample student answer. Your job is to ask OpenClaw to inspect those materials and help decide whether the lab is ready for students.

We will start with the simplest case: asking a model to reply to one prompt. Then we will slowly add more pieces: course files, workspace inspection, CLI tools, session history, and memory.

Do not worry about the word **harness** yet. First run the cells, read the outputs, and write down what changes from one step to the next. We will name the system pieces after you have seen them work.

## Section 1: Run OpenClaw On Course Lab Materials

We will build your understanding of OpenClaw one layer at a time.

First, you will connect OpenClaw to a model. Then you will ask it to run a small agent task. After that, you will inspect the course files yourself, ask the agent to use those files, ask it to use CLI tools, compare session behavior, read a memory note, and finally combine everything into a short course-assistant report.

By the end of this section, you should have seen OpenClaw:

1. call a configured model,
2. run a simple agent task,
3. answer from course files,
4. use CLI tools to inspect the workspace,
5. continue a session,
6. read a durable memory note,
7. combine those pieces into a short lab release report.

For each step, focus on two questions: what did you ask OpenClaw to do, and what evidence in the output shows how it did it?



### 1. Choose Your Model Provider

**What you will do:** choose how OpenClaw will reach a language model.

OpenClaw is not a model by itself. It needs a **Model Provider**: a backend that OpenClaw can send prompts to and receive model outputs from. For this lab, you only need one working provider path.

Here you are not choosing the agent runtime yet. You are choosing how OpenClaw reaches a model. Later, `agent --local` will use the configured model to run a full agent task.

**Choose one provider path before continuing.**

- DeepSeek is a hosted API provider. Use it only if you have your own API key and accept any billing.
- Ollama is a local model provider. It can run inside Colab or on your own machine, but GPU is recommended for later agent tasks.
- If your class uses another provider, vLLM, SGLang, or an OpenAI-compatible endpoint, you can adapt the notebook to that backend.

OpenClaw often writes models as `provider/model`, such as `deepseek/deepseek-v4-flash` or `ollama/qwen2.5-coder:7b`. That is the model reference syntax. The main idea for now is simpler: OpenClaw needs a provider before it can call a model.

**What to look for:** after setup, OpenClaw should list a model and the later Hello World call should return a short answer. If model setup fails, fix that before running the agent tasks.


In [ ]:
import getpass
import os
import pathlib
import shlex
import shutil
import subprocess
from typing import Iterable
REPO_URL = "https://github.com/SleepyLGod/openclaw-teaching.git"
REPO_DIR = pathlib.Path("/content/openclaw-teaching")
LAB_DIR = REPO_DIR / "labs" / "overall-harness"
WORKSPACE_SRC = LAB_DIR / "fundamentals-workspace"
OPENCLAW_WORKSPACE = pathlib.Path.home() / ".openclaw" / "workspace"
# Change this if you use another configured API provider/model.
MODEL_REF = "deepseek/deepseek-v4-flash"
# Default for Colab T4/local GPU: better for workspace + CLI-style agent tasks.
# If your runtime is CPU-only or too small, change this to "llama3.2:3b"
# or another model exposed through your Ollama provider.
OLLAMA_MODEL_NAME = "qwen2.5-coder:7b"
OLLAMA_MODEL_REF = f"ollama/{OLLAMA_MODEL_NAME}"
def section(title: str) -> None:
    line = "=" * len(title)
    print(f"\n{line}\n{title}\n{line}")
def run(cmd: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    print(f"\n$ {cmd}")
    proc = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return proc
def quote(text: str) -> str:
    return shlex.quote(text)
def show_file(path: pathlib.Path) -> None:
    print(f"\n--- {path} ---")
    print(path.read_text(encoding="utf-8"))
def show_files(paths: Iterable[pathlib.Path]) -> None:
    for path in paths:
        if path.is_file():
            show_file(path)


### 2. Set Up OpenClaw

**What you will do:** clone the teaching repo, install OpenClaw in this runtime, and copy the course lab materials into OpenClaw's workspace.

The workspace is the folder OpenClaw will use for this lab. It contains the files the agent will later inspect: `COURSE_BRIEF.md`, `FAQ.md`, `RUBRIC.md`, `LAB_RELEASE_CHECKLIST.md`, `MEMORY.md`, and a sample student answer.

This cell is also your first reading checkpoint. The notebook prints the Markdown files so you can inspect them yourself before asking the agent to use them. That gives you ground truth: later, you can compare the agent's answer against the actual file contents.

**What to look for:** read the printed file contents. Notice what each file is for. When the agent answers later, check whether it names the right files and uses details that really appear in those files.



In [ ]:
section("2. Set Up OpenClaw")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
_ = run(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")

print("Installing OpenClaw. This may take a minute.")
_ = run("curl -fsSL https://openclaw.ai/install.sh | bash -s -- --no-onboard", check=False)

os.environ["PATH"] = ":".join([
    str(pathlib.Path.home() / ".local" / "bin"),
    str(pathlib.Path.home() / ".openclaw" / "bin"),
    os.environ.get("PATH", ""),
])

_ = run("command -v openclaw || true", check=False)
_ = run("openclaw --version", check=False)
_ = run("openclaw doctor", check=False)

OPENCLAW_WORKSPACE.mkdir(parents=True, exist_ok=True)
for src in WORKSPACE_SRC.rglob("*"):
    if src.is_file():
        target = OPENCLAW_WORKSPACE / src.relative_to(WORKSPACE_SRC)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, target)

print("OpenClaw workspace target:", OPENCLAW_WORKSPACE)
print("\nCourse lab files copied into the workspace:")
workspace_files = sorted(
    path for path in OPENCLAW_WORKSPACE.rglob("*")
    if path.is_file() and path.name in {
        "COURSE_BRIEF.md",
        "LAB_RELEASE_CHECKLIST.md",
        "RUBRIC.md",
        "FAQ.md",
        "MEMORY.md",
        "sample_student_answer.md",
    }
)
show_files(workspace_files)


### 3. Configure A Model Provider

**What you will do:** run the DeepSeek provider cell, the Ollama provider cell, or another provider setup your instructor gives you.

**Choose one provider path before continuing.** You do not need to run both DeepSeek and Ollama. DeepSeek is usually more stable for a live classroom demo if you have a key. Ollama is useful for seeing a local-model path, but full agent tasks can be slow on CPU.

If you run both provider setup cells, the later one decides which model the following OpenClaw tasks use. That is useful for comparison, but it can also be confusing. For your first run, pick one path and stay with it.

**What to look for:** OpenClaw should show the configured provider and model. If you choose the Ollama cell, the notebook switches later tasks to the Ollama model automatically.


#### 3A. Optional API Route: DeepSeek

**What you will do:** connect OpenClaw to DeepSeek using your own API key.

If you are in Colab, the notebook first checks Colab Secrets for `DEEPSEEK_API_KEY`. If it is not there, it asks for hidden input. The key should not be printed in the notebook output.

**What to look for:** the model list should include DeepSeek models. If you do not have a key, skip this route and use a local or instructor-provided route instead.



In [ ]:
section("3A. Configure DeepSeek API route")
print("Model reference:", MODEL_REF)
print("This cell does not print your API key.")

if not os.environ.get("DEEPSEEK_API_KEY"):
    try:
        from google.colab import userdata
        secret = userdata.get("DEEPSEEK_API_KEY")
    except Exception:
        secret = None

    if secret:
        os.environ["DEEPSEEK_API_KEY"] = secret
        print("Loaded DEEPSEEK_API_KEY from Colab Secrets.")
    else:
        os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Enter DEEPSEEK_API_KEY: ")
else:
    print("DEEPSEEK_API_KEY already exists in the environment.")

_ = run('openclaw onboard --non-interactive --mode local --auth-choice deepseek-api-key --deepseek-api-key "$DEEPSEEK_API_KEY" --skip-health --accept-risk', check=False)
_ = run("openclaw models list --all --provider deepseek", check=False)
_ = run(f"openclaw models set {shlex.quote(MODEL_REF)}", check=False)


#### 3B. Optional Local Provider: Ollama

**What you will do:** install Ollama in this Colab environment, start the local server, pull a model, and tell OpenClaw to use it.

The default model is `qwen2.5-coder:7b`. It is a better fit for workspace and CLI-style agent tasks than a very small model, and it usually fits a Colab T4 GPU. If your runtime has less memory, is CPU-only, or feels too slow, change `OLLAMA_MODEL_NAME` in the next code cell to a smaller model such as `llama3.2:3b`.

Local models are useful for learning, but they are not guaranteed to behave like stronger hosted models. A small local model may complete Hello World and still struggle when the full agent task includes files, tools, session history, and longer instructions. Treat that as an observation about local-model capacity, not as a failed lab.

**What to look for:** Ollama should start, the model should download, and OpenClaw should set the Ollama model for later cells.


In [ ]:
section("3B. Configure Ollama local route")
print("Ollama model:", OLLAMA_MODEL_NAME)
print("If you change OLLAMA_MODEL_NAME, rerun this cell before the OpenClaw tasks.")
os.environ["OLLAMA_API_KEY"] = "ollama-local"
_ = run("apt-get update", check=False)
_ = run("apt-get install -y zstd curl ca-certificates", check=False)
_ = run("curl -fsSL https://ollama.com/install.sh | sh", check=False)
_ = run("pgrep -x ollama >/dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)", check=False)
_ = run("sleep 5 && curl -s http://127.0.0.1:11434/api/tags || true", check=False)
_ = run(f"ollama pull {shlex.quote(OLLAMA_MODEL_NAME)}", check=False)
_ = run("openclaw models list --all --provider ollama", check=False)
_ = run(f"openclaw models set {shlex.quote(OLLAMA_MODEL_REF)}", check=False)
MODEL_REF = OLLAMA_MODEL_REF
print("Notebook model for later tasks:", MODEL_REF)


### 4. LLM Hello World

**What you will do:** send one short prompt directly to the selected model.

This is the smallest run in the notebook. OpenClaw sends exactly the prompt in the code cell to the selected Model Provider and prints the model's reply. No course files, tools, session history, or memory notes are needed for this step.

**What to look for:** the output should contain `OpenClaw is connected.` or something very close. This tells you the provider and model are connected. It does not yet show what makes an agent task different.


In [ ]:
section("4. LLM Hello World")
prompt = "Reply with exactly: OpenClaw is connected."
print("Command path: openclaw infer model run")
print("Model reference:", MODEL_REF)
_ = run(f"openclaw infer model run --local --model {shlex.quote(MODEL_REF)} --prompt {quote(prompt)} --json", check=False)


### 5. Agent Hello World

**What you will do:** run your first OpenClaw agent task.

The message is still short, but this time OpenClaw starts an agent turn with a session key and the workspace prepared earlier. You are now asking the agent workbench to handle a task, not only asking a model to complete one prompt.

**What to look for:** compare this output with the direct model call. You may see extra runtime messages, and the answer may sound more like a course assistant. At this point, the task is still simple; the next steps ask the agent to use files and tools.



In [ ]:
section("5. Agent Hello World")
message = "You are helping with an AI Agent course lab. In two sentences, say what kind of work you can help with in this workspace."
print("Command path: openclaw agent --local")
print("Session key: fundamentals-agent-hello")
_ = run(f"openclaw agent --local --session-key fundamentals-agent-hello --message {quote(message)}", check=False)


### 6. Ask About The Course Files

**What you will do:** ask the agent to read `COURSE_BRIEF.md` and `FAQ.md` before answering.

This is the first file-grounded task. The agent should use the lab workspace rather than answer from general knowledge.

**What to look for:** check whether the answer mentions concrete details from the course brief or FAQ. Because you already read the printed files in Step 2, you have a ground-truth reference. If the answer only gives generic statements about AI agents, note that as a behavior to investigate.



In [ ]:
section("6. Ask About The Course Files")
message = "Read COURSE_BRIEF.md and FAQ.md. Summarize what this lab is about and list two questions students may ask."
print("Files expected to matter: COURSE_BRIEF.md, FAQ.md")
_ = run(f"openclaw agent --local --session-key fundamentals-course-files --message {quote(message)}", check=False)


### 7. Use CLI Tools To Inspect The Workspace

**What you will do:** ask OpenClaw to inspect the lab workspace with CLI tools.

In this lab, a **tool** means an action OpenClaw can run for the agent. The model can request a tool call, but it does not directly own your filesystem and does not run shell commands by itself. OpenClaw decides which tools are available, runs the action, and returns results to the agent.

This is a real tool-use step, not fake agent code and not a calculator toy. The task is similar to what a TA might do before publishing lab materials: check that the required files exist and decide which file should be inspected first.

**What to look for:** the answer should name the files it found. If you see a tool-policy line, do not treat it as an error by default; it means OpenClaw is shaping which tools are available for this agent task.



In [ ]:
section("7. Use CLI Tools To Inspect The Workspace")
message = "Use CLI tools to inspect the lab workspace. Check whether COURSE_BRIEF.md, LAB_RELEASE_CHECKLIST.md, RUBRIC.md, FAQ.md, MEMORY.md, and submissions/sample_student_answer.md exist. Then report which files are present and which file you would inspect first before publishing the lab."
print("Expected action: use runtime-provided CLI/filesystem tools to inspect workspace files.")
_ = run(f"openclaw agent --local --session-key fundamentals-cli-tools --message {quote(message)}", check=False)


### 8. Continue The Same Session

**What you will do:** give the agent a temporary review preference, then ask it to review a sample student answer in the same session.

After that, you will ask a similar review question with a fresh session key.

**What to look for:** in the same session, the agent should use the review preference you just gave it. In the fresh session, that preference should not automatically carry over unless it comes from another source such as a workspace file or memory note.



In [ ]:
section("8. Continue The Same Session")
review_session = "fundamentals-review-session"
turn_1 = "For this session, when you review student answers, focus first on whether the answer cites observable OpenClaw behavior."
turn_2 = "Review submissions/sample_student_answer.md using the review preference I just gave you."
fresh_turn = "Review submissions/sample_student_answer.md. What review preference are you using?"

print("Same session key:", review_session)
_ = run(f"openclaw agent --local --session-key {review_session} --message {quote(turn_1)}", check=False)
_ = run(f"openclaw agent --local --session-key {review_session} --message {quote(turn_2)}", check=False)

print("\nFresh session key: fundamentals-review-fresh")
_ = run(f"openclaw agent --local --session-key fundamentals-review-fresh --message {quote(fresh_turn)}", check=False)


### 9. Use A Simple Memory Note

**What you will do:** ask the agent to read `MEMORY.md`, a small durable note in the workspace.

This is not a deep memory-system lab. For now, treat `MEMORY.md` as a simple example of a durable preference written down outside the immediate chat turn.

**What to look for:** the answer should explain the TA's feedback preference from `MEMORY.md`. Compare this with the temporary session preference from the previous step.



In [ ]:
section("9. Use A Simple Memory Note")
message = "Read MEMORY.md and tell me the durable TA preference for feedback style."
print("File expected to matter: MEMORY.md")
_ = run(f"openclaw agent --local --session-key fundamentals-memory-note --message {quote(message)}", check=False)


### 10. Complete A Small Course Assistant Task

**What you will do:** ask OpenClaw for a short release-readiness report.

This final task combines the course files, FAQ, rubric, release checklist, memory note, current session preference, and workspace inspection into one useful answer.

**What to look for:** the exact wording will vary by model. A strong answer should mention which files or tools informed it. If the answer sounds plausible but gives no evidence, note that as a weakness.



In [ ]:
section("10. Complete A Small Course Assistant Task")
message = "You are preparing this OpenClaw lab for students. Use the course brief, FAQ, rubric, release checklist, memory note, and the current session preference. Produce a short release-readiness report with: 1. what the lab teaches, 2. what students must observe, 3. one likely student confusion, 4. one suggested improvement before publishing. Mention which files or tools informed your answer."
print("Session key: fundamentals-review-session")
print("Expected inputs: course files, memory note, prior review preference, and tool-visible workspace state.")
_ = run(f"openclaw agent --local --session-key fundamentals-review-session --message {quote(message)}", check=False)


## Section 2: What Was Around The Model?

We have already seen the behavior. Now we name the pieces.

In Section 1, the direct model call was small: one prompt went to one model and came back as one answer. The later OpenClaw runs felt different because the model was no longer working alone. It had a workspace, course files, tools, action rules, session history, and a memory note around it.

People use the word **harness** for this kind of setup, but the word is used at different levels. This section explains the idea first, then maps it back to what you just ran in OpenClaw.

A short version: an agent run is model + context + tools + state + policy + loop + evidence.

You do not need to run new code here. Read this as a guided explanation of the runs you just performed.


### 1. From The Demo To The Concept

Start with a concrete comparison.

If you ask for a course release report through a direct model call, the model only sees the prompt you typed. It does not automatically know your course files, your checklist, your session preference, or your memory note.

If you ask for the same report through an OpenClaw agent run, OpenClaw can prepare a richer work setting first.

```text
Direct model call

one prompt -> selected model -> answer


OpenClaw agent run

task
  -> workspace files
  -> tools and action rules
  -> session history
  -> memory note
  -> selected model
  -> answer with evidence
```

The model did not become a different kind of intelligence. The work setting around the model changed.

This matters because agent problems are often setup problems. The task may be vague. A file may be missing. A tool may be unavailable. The session may contain old instructions. The answer may not cite evidence. Harness engineering is the practice of finding and improving those setup pieces.


### 2. Reading The Harness Picture

The word **harness** is broad. Martin Fowler's article uses this picture to show that the meaning changes depending on where you draw the boundary.

![Fowler bounded-context harness diagram](https://martinfowler.com/articles/harness-engineering/harness-bounded-contexts.png)

*Source: [Martin Fowler, Harness Engineering](https://martinfowler.com/articles/harness-engineering.html).*

Read the picture from the inside out:

1. **Inner model.** This is the language model itself. It predicts and writes text.
2. **OpenClaw-built harness.** This is what OpenClaw adds around the model: runtime, prompt assembly, workspace handling, tool execution, action rules, and session handling.
3. **Our course harness.** This is what we add around OpenClaw for this class: course files, rubrics, release checklists, memory notes, review questions, and future evals.

So in this lab, “overall harness” does not mean one magic component. It means the whole prepared work setting around the model. Some of it is built by OpenClaw. Some of it is built by us for the course task.


### 3. A Definition We Will Use

There is no single universal definition of **harness**. We will use two sources to set the boundary.

Martin Fowler notes that one broad usage is: **“everything in an AI agent except the model itself.”** That is useful, but very broad. It includes many things: instructions, files, tools, tests, logs, review steps, and feedback loops.

OpenClaw uses the word more narrowly in its runtime documentation: **“A harness is the implementation that provides an agent runtime.”** In other words, OpenClaw also has a source-code meaning for harness. We will study that implementation later.

For this lab, use the broad teaching meaning:

> A harness is the setup around the model that helps it do a task reliably: files, instructions, tools, action rules, session history, memory notes, checks, and feedback loops.

Keep these five things separate:

| Do not confuse... | Meaning in this lab |
| --- | --- |
| Model | DeepSeek, an Ollama model, or another LLM that generates text |
| Model Provider | The route OpenClaw uses to reach the model, such as DeepSeek API or Ollama |
| Prompt | One piece of input sent to the model |
| Agent run | The full OpenClaw task turn started by `openclaw agent --local` |
| Harness | The files, tools, rules, session, memory, and checks around that run |

The useful habit is simple: when an agent gives a weak answer, do not only ask whether the model is strong enough. Ask what part of the work setting was weak.


### 4. Before The Agent Acts: Guides

Before an agent works, we can give it guides. In Fowler's article, **Guides** is the explicit term: guides are feedforward controls that steer the agent before it acts.

In plain language, a guide is anything the agent can use before doing the task: instructions, reference docs, examples, tool descriptions, or project rules.

![Fowler harness overview diagram](https://martinfowler.com/articles/harness-engineering/harness-overview.png)

*Source: [Martin Fowler, Harness Engineering](https://martinfowler.com/articles/harness-engineering.html). The left side shows guides before work starts. The right side shows feedback sensors after work happens.*

In Section 1, these were guides:

| Guide | How it helped |
| --- | --- |
| `COURSE_BRIEF.md` | Explained what the lab is about |
| `FAQ.md` | Gave likely student questions |
| `RUBRIC.md` | Described what a good answer should include |
| `LAB_RELEASE_CHECKLIST.md` | Gave publishing criteria |
| `MEMORY.md` | Stored a durable TA preference |
| Tool descriptions | Told the model what actions OpenClaw can run |
| The user task | Told the agent what to produce now |

This is why harness engineering is bigger than prompt engineering. A prompt is one guide. A real agent run often needs many guides, and we must decide which ones belong in the workspace, which ones belong in memory, which ones belong in a tool description, and which ones belong in the user task.


### 5. After The Agent Runs: What Can We Check?

After an agent run, we need evidence. We need to know what files were used, what tools were available, whether session history mattered, and whether the answer can be checked against the workspace.

Fowler's article calls these feedback controls **Sensors**. In this lab, read that word simply: a sensor is a check or observation point, and a signal is the evidence it leaves behind.

In Section 1, these were simple checks and signals:

| Sensor or check | Signal to read |
| --- | --- |
| Notebook prints workspace files | Can you compare the answer with the real course files? |
| CLI inspection result | Do the expected workspace files exist? |
| Tool-policy output | Which actions did OpenClaw allow or remove? |
| Same-session vs fresh-session comparison | Did session history affect the answer? |
| Final report evidence | Did the answer cite files or tools, or was it generic? |

Later chapters will add stronger checks: evals, guardrail checks, test cases, traces, review agents, and logs.

The main point is practical: an agent run should leave evidence. If you cannot tell what happened, you cannot improve the harness.


### 6. Two Ways To Check The Agent's Work

Some checks are easy for a program. Some checks require judgment.

Ask yourself: can a normal program check this, or does someone need to judge the meaning?

| Question | Best check | Lab example |
| --- | --- | --- |
| Does `FAQ.md` exist? | Program check | CLI inspection can list files |
| Did the notebook code parse? | Program check | `ast.parse` can validate Python cells |
| Did the answer cite observable OpenClaw behavior? | Judgment check | A human or review agent reads the answer |
| Is the explanation clear for students? | Judgment check | TA/professor review or model-as-reviewer |

Use program checks whenever they fit. They are fast and reliable. Use judgment checks when the question is about meaning, clarity, or pedagogy. Fowler calls these **computational** and **inferential** checks; the names matter less than choosing the right kind of check.

A strong harness usually combines both. For example, first check that the files exist; then review whether the answer used them well.


### 7. When The Output Is Weak, What Should We Change?

Harness engineering becomes useful when something goes wrong. Instead of saying “try again” or “use a bigger model,” inspect the setup.

Use these examples from the lab:

| If you see this output... | First question to ask | Smallest useful change |
| --- | --- | --- |
| The answer is generic | Did we give enough course-specific guidance? | Improve `COURSE_BRIEF.md`, the user task, or the rubric |
| The agent misses a required file | Is the file actually in the workspace? | Check setup cell output and file names |
| The agent cannot inspect files | Are the needed tools available? | Check tool output and action rules |
| A previous instruction leaks into a new run | Are we using the same `--session-key`? | Use a fresh session key or reset the session |
| The final report sounds good but cites no evidence | Did we ask for evidence? | Add “mention which files or tools informed your answer” |

This is the steering loop:

```text
run the agent
  -> inspect the result
  -> identify the weak part of the setup
  -> improve the guide, tool, rule, session, memory, or check
  -> run again
```

OpenAI's harness engineering post describes a similar shift: engineers increasingly spend time designing agent environments, stating intent clearly, and building feedback loops. In this lab, the loop is small, but the habit is the same.


### 8. What Kind Of Harness Are We Studying?

Fowler later separates harness goals into maintainability, architecture fitness, and behavior. Those categories are useful for coding agents, but this first OpenClaw lab does not need students to memorize them.

For now, we are studying the **overall shape** of a harness:

- guides before the agent acts,
- tools and rules while it acts,
- session and memory across turns,
- checks and evidence after it acts,
- a loop for improving the setup.

Later chapters will go deeper. Guardrails will focus on action rules. Eval will focus on checks, evidence, and review. Memory will focus on durable state. Skills will focus on reusable guidance. Source walkthroughs will show how OpenClaw implements these pieces.


### 9. OpenClaw Checkpoint

Use this table to diagnose the runs from Section 1. The goal is not to memorize a term. The goal is to know where to look first.

| If this step has a problem... | Inspect this first | Why |
| --- | --- | --- |
| LLM Hello World | Model Provider + Model | The direct call only proves the selected model can answer |
| Agent Hello World | Agent run setup | OpenClaw should start a full task turn |
| Ask About The Course Files | Workspace and task wording | The agent needs the right files and a clear request |
| Use CLI Tools To Inspect The Workspace | Tools and action rules | OpenClaw must expose and allow the needed actions |
| Tool-policy output is surprising | Action rules | Tool availability is being shaped by policy |
| Same-session behavior is surprising | Session key and history | The same key carries short-term task history |
| Fresh session still remembers old context | Session reset or hidden workspace/memory source | Something outside the immediate turn may be influencing the answer |
| Memory note is ignored | `MEMORY.md` and task wording | The agent may need a clearer instruction to read it |
| Final report has no evidence | Feedback requirement | Ask the agent to mention which files or tools informed the answer |

After this section, you should be able to say this in your own words:

> OpenClaw did more than call a model. It prepared files, tools, action rules, session history, memory notes, and evidence around the model. When an agent output is weak, check those pieces before you only blame the model.

Optional reading:

- Martin Fowler: https://martinfowler.com/articles/harness-engineering.html
- OpenAI: https://openai.com/index/harness-engineering/
